In [ ]:
INPUT_CSV   = 'results.csv'
OUTPUT_XLSX = 'scorecard.xlsx'

# Business day windows 
ACK_DEADLINE_BD       = 5    # NH RSA 91-A scores −5
FULFILL_DEADLINE_BD   = 25   # Past this acknowledged-but-pending scores −3
IMMEDIATE_BD          = 10   # Completed within this (≤) scores +5
DELAYED_FULFILLED_BD  = 60   # Completed past this (>) scores +3
                             # Between IMMEDIATE_BD and DELAYED_FULFILLED_BD scores +4

AUTO_EXCLUDE_CATEGORIES   = {'RFP List'}
AUTO_EXCLUDE_STATUS_CODES = {'fix'}

COUNTERPART_OF = {
    'Berlin School District/SAU 3':                          'Berlin',
    'SAU6, serving the communities of Claremont and Unity':  'Claremont',
    'Concord School District/SAU8':                          'Concord',
    'Dover School District/SAU11':                           'Dover',
    'Franklin School District/SAU18':                        'Franklin',
    'Manchester School District':                            'Manchester',
    'Keene School District/SAU29':                           'Keene',
    'Laconia School District/SAU30':                         'Laconia',
    'Lebanon School District/SAU88':                         'Lebanon',
    'Nashua School District':                                'Nashua',
    'Portsmouth School District':                            'Portsmouth',
    'Rochester School District':                             'Rochester',
    'Somersworth School District/SAU56':                     'Somersworth',
    'Contoocook Valley School District/SAU1':                'Peterborough',
    'Shaker Regional School District/SAU80':                 'Belmont',
    'SAU70':                                                 'Hanover',
    'SAU20':                                                 'Gorham',
    'SAU67':                                                 'Bow',
    'Hopkinton School District':                             'Hopkinton',
    'Derry Cooperative School District':                     'Derry',
    'Salem School District/SAU57':                           'Salem',
    'Monadnock Regional School District':                    'Troy',
    'Oyster River Cooperative School District':              'Durham',
}

# these looked messy with the muckrock request agency names as-is
DISTRICT_DISPLAY_NAME = {
    'SAU67':                                                 'Bow School District/SAU67',
    'SAU6, serving the communities of Claremont and Unity':  'Claremont School District/SAU6',
    'Derry Cooperative School District':                     'Derry Cooperative School District/SAU10',
    'SAU20':                                                 'Gorham School District/SAU20',
    'SAU70':                                                 'Interstate School District/SAU70',
    'Manchester School District':                            'Manchester School District/SAU37',
    'Monadnock Regional School District':                    'Monadnock Regional School District/SAU93',
    'Nashua School District':                                'Nashua School District/SAU 42',
    'Oyster River Cooperative School District':              'Oyster River Cooperative School District/SAU5',
    'Portsmouth School District':                            'Portsmouth School District/SAU52',
    'Rochester School District':                             'Rochester School District/SAU54',
    'Hopkinton School District':                            'Hopkinton School District/SAU66',
}


In [ ]:
import csv, re
from collections import Counter
from datetime import date, datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

with open(INPUT_CSV, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

RAW_DISTRICT_NAMES = set(COUNTERPART_OF.keys())

def display_of(raw_name):
    return DISTRICT_DISPLAY_NAME.get(raw_name, raw_name)

DISTRICT_DISPLAY_NAMES   = {display_of(raw) for raw in RAW_DISTRICT_NAMES}
DISTRICT_TO_HOST_TOWN    = {display_of(raw): host for raw, host in COUNTERPART_OF.items()}
TOWN_TO_DISTRICT_DISPLAY = {host: display_of(raw) for raw, host in COUNTERPART_OF.items()}

_TRAIL_PAREN = __import__("re").compile(r"\s*\([^)]*\)\s*$")
def category(t): return _TRAIL_PAREN.sub("", t or "").strip()

def entity_of(row):
    a = (row.get('Agency Name') or '').strip()
    if a in RAW_DISTRICT_NAMES:
        return display_of(a)
    return row['Jurisdiction']

def kind_of(entity):
    return 'district' if entity in DISTRICT_DISPLAY_NAMES else 'town'

def counterpart_of(entity):
    if entity in DISTRICT_DISPLAY_NAMES:
        return DISTRICT_TO_HOST_TOWN[entity]
    return TOWN_TO_DISTRICT_DISPLAY.get(entity)

def normalized_tags(r): return (r.get('Tags') or '').lower().replace('nh-in-person', 'nh-inperson')
def has_tag(r, frag): return frag in normalized_tags(r)

def _parse_dt(s):
    if not s: return None
    try: return datetime.strptime(s.split()[0], '%Y-%m-%d').date()
    except ValueError: return None

def business_days_between(a, b):
    if not a or not b or b <= a: return 0
    total = (b - a).days
    weeks, rem = divmod(total, 7)
    n = weeks * 5
    wd = a.weekday()
    for d in range(1, rem + 1):
        if (wd + d) % 7 < 5: n += 1
    return n

def business_days_open(row, today=None):
    s = _parse_dt(row.get('Date Submitted'))
    d = _parse_dt(row.get('Date Done'))
    return business_days_between(s, d or (today or date.today()))

def auto_score(row, today=None):
    code = (row.get('Status Code') or '').strip()
    bd   = business_days_open(row, today)
    if has_tag(row, 'privacy'): return -2
    if code == 'done':
        if has_tag(row, 'nh-inperson') or has_tag(row, 'nh-citizen'): return 2
        if bd > DELAYED_FULFILLED_BD: return 3
        if bd <= IMMEDIATE_BD: return 5
        return 4
    if code == 'no_docs':  return 0
    if code == 'fix':      return None
    if code == 'partial':  return None
    if code == 'rejected': return -4
    if code == 'ack':                          return -5 if bd > ACK_DEADLINE_BD else None
    if code in ('processed', 'submitted'):     return -3 if bd > FULFILL_DEADLINE_BD else None
    return None

def is_auto_excluded(row):
    return (
        category(row['Title']) in AUTO_EXCLUDE_CATEGORIES
        or (row.get('Status Code') or '').strip() in AUTO_EXCLUDE_STATUS_CODES
    )

print(f'Loaded {len(rows)} requests from {INPUT_CSV}')

unknown = sorted({
    (row.get('Agency Name') or '').strip()
    for row in rows
    if 'school district' in (row.get('Agency Name') or '').lower() or 'sau' in (row.get('Agency Name') or '').lower()
    if (row.get('Agency Name') or '').strip() not in RAW_DISTRICT_NAMES
})
if unknown:
    print(f'unmapped agency names:')
    for u in unknown: print(f'    {u!r}')

codes = Counter((r.get('Status Code') or '').strip() for r in rows)
print(f'status codes:')
for code, n in sorted(codes.items()):
    print(f'  {code or "(blank)":<12} {n:4d}')



In [ ]:
wb = Workbook()
ws = wb.active
ws.title = 'Requests'

BOLD       = Font(bold=True)
HDR_FILL   = PatternFill('solid', fgColor='EEF3FA')
ADDED_FILL = PatternFill('solid', fgColor='FDF8E0')  
LINK       = Font(color='0563C1', underline='single')

CSV_COLS = list(rows[0].keys()) if rows else []

# scoring columns
ADDED_COLS = ['Entity', 'Kind', 'Counterpart', 'Auto Score', 'Manual Score', 'Effective', 'Exclude', 'Notes']

headers = CSV_COLS + ADDED_COLS
ws.append(headers)
for c in ws[1]:
    c.font = BOLD
    c.fill = HDR_FILL
    c.alignment = Alignment(wrap_text=True, vertical='center')

COL = {h: i + 1 for i, h in enumerate(headers)}
AUTO_COL    = COL['Auto Score']
MANUAL_COL  = COL['Manual Score']
EFF_COL     = COL['Effective']
EXCLUDE_COL = COL['Exclude']

today = date.today()

for ridx, row in enumerate(rows, start=2):
    for key in CSV_COLS:
        v = row.get(key, '')
        cell = ws.cell(row=ridx, column=COL[key], value=v)
        if key == 'MuckRock URL' and v:
            cell.hyperlink = v
            cell.font = LINK

    entity = entity_of(row)
    kind   = kind_of(entity)
    cp     = counterpart_of(entity)
    auto   = auto_score(row, today)
    excl   = is_auto_excluded(row)

    ws.cell(row=ridx, column=COL['Entity']).value      = entity
    ws.cell(row=ridx, column=COL['Kind']).value        = kind
    ws.cell(row=ridx, column=COL['Counterpart']).value = cp
    if auto is not None:
        ws.cell(row=ridx, column=AUTO_COL).value = auto
    ws.cell(row=ridx, column=EXCLUDE_COL).value = excl

    eff_formula = (
        f'=IF({get_column_letter(EXCLUDE_COL)}{ridx}=TRUE,"",'
        f'IF(ISNUMBER({get_column_letter(MANUAL_COL)}{ridx}),'
        f'{get_column_letter(MANUAL_COL)}{ridx},'
        f'{get_column_letter(AUTO_COL)}{ridx}))'
    )
    ws.cell(row=ridx, column=EFF_COL).value = eff_formula

    for h in ADDED_COLS:
        ws.cell(row=ridx, column=COL[h]).fill = ADDED_FILL


In [ ]:
ws2 = wb.create_sheet('Entities')
ent_headers = ['Entity', 'Kind', 'Counterpart', 'Request Count', 'Avg Score', 'Summary', 'Agency Response']
ws2.append(ent_headers)
for c in ws2[1]:
    c.font = BOLD
    c.fill = HDR_FILL

entities = {}
for row in rows:
    e = entity_of(row)
    if e not in entities:
        entities[e] = {'kind': kind_of(e), 'counterpart': counterpart_of(e)}

sorted_entities = sorted(entities.items(), key=lambda kv: (kv[1]['kind'] == 'district', kv[0]))

last_row = len(rows) + 1
ent_rng = f"Requests!${get_column_letter(COL['Entity'])}$2:${get_column_letter(COL['Entity'])}${last_row}"
eff_rng = f"Requests!${get_column_letter(EFF_COL)}$2:${get_column_letter(EFF_COL)}${last_row}"

for ridx, (entity, info) in enumerate(sorted_entities, start=2):
    ws2.cell(row=ridx, column=1, value=entity)
    ws2.cell(row=ridx, column=2, value=info['kind'])
    ws2.cell(row=ridx, column=3, value=info['counterpart'])
    ws2.cell(row=ridx, column=4, value=f'=COUNTIF({ent_rng},A{ridx})')
    ws2.cell(row=ridx, column=5, value=f'=IFERROR(AVERAGEIFS({eff_rng},{ent_rng},A{ridx}),"")')
    ws2.cell(row=ridx, column=6).fill = ADDED_FILL
    ws2.cell(row=ridx, column=7).fill = ADDED_FILL

ws2.freeze_panes = 'B2'
WIDTHS2 = {'Entity': 30, 'Kind': 10, 'Counterpart': 30, 'Request Count': 12, 'Avg Score': 12, 'Summary': 70, 'Agency Response': 70}
for i, h in enumerate(ent_headers, start=1):
    ws2.column_dimensions[get_column_letter(i)].width = WIDTHS2.get(h, 14)
for row in ws2.iter_rows(min_row=2):
    row[5].alignment = Alignment(wrap_text=True, vertical='top')
    row[6].alignment = Alignment(wrap_text=True, vertical='top')

wb.save(OUTPUT_XLSX)
n_towns     = sum(1 for _, info in sorted_entities if info['kind'] == 'town')
n_districts = sum(1 for _, info in sorted_entities if info['kind'] == 'district')
print(f'{OUTPUT_XLSX}: {len(rows)} requests across {n_towns} towns + {n_districts} districts')
